In [1]:
# 大模型没有非sft的训练方式
import json
import numpy as np
from transformers import BertTokenizer
import torch

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 初始化 tokenizer
tokenizer = BertTokenizer.from_pretrained('/Users/bowie/Documents/muti-model/bert-base-chinese')

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
# 假设你的数据是一个 JSON 字符串
data = '{"text": "首先我们先来关心南斯拉夫总统大选的状况。", "entities": [{"start_idx": 8, "end_idx": 12, "entity_text": "南斯拉夫", "entity_label": "GPE"}]}'
data = json.loads(data)

# 文本和实体
text = data['text']
entities = data['entities']

# 对文本进行分词
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, add_special_tokens=True)  # 带有特殊标记的 token IDs
labels = ['O'] * len(tokens)  # 初始化标签为 'O'


In [4]:
tokens

['首',
 '先',
 '我',
 '们',
 '先',
 '来',
 '关',
 '心',
 '南',
 '斯',
 '拉',
 '夫',
 '总',
 '统',
 '大',
 '选',
 '的',
 '状',
 '况',
 '。']

In [7]:
# 根据实体信息更新标签
for entity in entities:
    start_idx = entity['start_idx']
    end_idx = entity['end_idx']
    
    # 找到对应的 token 索引
    start_token_idx = len(tokenizer.tokenize(text[:start_idx]))  # 起始 token 的索引
    end_token_idx = len(tokenizer.tokenize(text[:end_idx]))      # 结束 token 的索引
    
    # 更新标签
    for i in range(start_token_idx, end_token_idx):
        if i == start_token_idx:
            labels[i] = 'B-' + entity['entity_label']  # 开始 token
        else:
            labels[i] = 'I-' + entity['entity_label']  # 中间 token


In [8]:
# 将标签转换为数值（例如用字典映射）
label_map = {'O': 0, 'B-GPE': 1, 'I-GPE': 2}
label_ids = [label_map[label] for label in labels]

In [9]:
label_ids

[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:
# 转换为 PyTorch 张量
input_ids = torch.tensor(token_ids).unsqueeze(0)  # 添加 batch 维度
attention_mask = (input_ids != 0).long()  # 生成注意力掩码
label_tensor = torch.tensor(label_ids).unsqueeze(0)  # 添加 batch 维度

# 现在你可以使用 input_ids, attention_mask, 和 label_tensor 来微调你的模型